##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, Conv2D, MaxPooling2D, Flatten, LSTM, Dense, Dropout

In [2]:
DATASET_PATH = "/content/UCF11_updated_mpg.rar"

CLASSES = [
    "basketball",
    "biking",
    "tennis_swing"
]

IMG_SIZE = 64
SEQUENCE_LENGTH = 20

In [3]:
def extract_frames(video_path):
    frames = []
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    skip = max(total_frames // SEQUENCE_LENGTH, 1)

    for i in range(SEQUENCE_LENGTH):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * skip)
        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    while len(frames) < SEQUENCE_LENGTH:
        frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3)))

    return np.array(frames)

In [5]:
!apt-get install unrar
!unrar x "/content/UCF11_updated_mpg.rar" "/content/"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/UCF11_updated_mpg.rar

Creating    /content/UCF11_updated_mpg                                OK
Creating    /content/UCF11_updated_mpg/basketball                     OK
Creating    /content/UCF11_updated_mpg/basketball/v_shooting_01       OK
Extracting  /content/UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_01.mpg       0%  OK 
Extracting  /content/UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_02.mpg       0%  OK 
Extracting  /content/UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_03.mpg       0%  OK 
Extracting  /content/UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_04.mpg       0%  OK 
Extract

In [6]:
import os
print(os.listdir("/content"))

['.config', 'UCF11_updated_mpg.rar', 'UCF11_updated_mpg', 'sample_data']


In [7]:
DATASET_PATH = "/content/UCF11_updated_mpg"

In [9]:
import os
print(os.listdir("/content/UCF11_updated_mpg"))

['golf_swing', 'basketball', 'biking', 'diving']


In [12]:
CLASSES = [
    "basketball",
    "biking",
    "diving"
]

In [13]:
for class_name in CLASSES:
    class_path = os.path.join(DATASET_PATH, class_name)
    print(class_name, "folders:", len(os.listdir(class_path)))

basketball folders: 26
biking folders: 26
diving folders: 26


In [14]:
X = []
y = []

for class_index, class_name in enumerate(CLASSES):
    class_path = os.path.join(DATASET_PATH, class_name)

    for folder in os.listdir(class_path):
        folder_path = os.path.join(class_path, folder)

        if not os.path.isdir(folder_path):
            continue

        for video_name in os.listdir(folder_path):
            video_path = os.path.join(folder_path, video_name)

            if video_name.endswith((".mpg", ".avi", ".mp4")):
                frames = extract_frames(video_path)
                X.append(frames)
                y.append(class_index)

X = np.array(X)
y = to_categorical(np.array(y), num_classes=len(CLASSES))

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (442, 20, 64, 64, 3)
y shape: (442, 3)


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (353, 20, 64, 64, 3)
Test: (89, 20, 64, 64, 3)


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, Conv2D, MaxPooling2D, Flatten, LSTM, Dense, Dropout

model = Sequential()

model.add(TimeDistributed(
    Conv2D(32, (3, 3), activation="relu"),
    input_shape=(20, 64, 64, 3)
))
model.add(TimeDistributed(MaxPooling2D((2, 2))))

model.add(TimeDistributed(Conv2D(64, (3, 3), activation="relu")))
model.add(TimeDistributed(MaxPooling2D((2, 2))))

model.add(TimeDistributed(Flatten()))

model.add(LSTM(64))
model.add(Dropout(0.5))

model.add(Dense(64, activation="relu"))
model.add(Dense(3, activation="softmax"))

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 20, 62, 62, 32) │           896 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 20, 31, 31, 32) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 20, 29, 29, 64) │        18,496 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 20, 14, 14, 64) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 20, 12544)      │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │     3,227,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,251,651 (12.40 MB)

 Trainable params: 3,251,651 (12.40 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=8,
    batch_size=8
)

Epoch 1/8
45/45 ━━━━━━━━━━━━━━━━━━━━ 147s 3s/step - accuracy: 0.6374 - loss: 0.7990 - val_accuracy: 0.7753 - val_loss: 0.5485
Epoch 2/8
45/45 ━━━━━━━━━━━━━━━━━━━━ 137s 3s/step - accuracy: 0.8725 - loss: 0.3876 - val_accuracy: 0.8427 - val_loss: 0.4399
Epoch 3/8
45/45 ━━━━━━━━━━━━━━━━━━━━ 143s 3s/step - accuracy: 0.8697 - loss: 0.3588 - val_accuracy: 0.7753 - val_loss: 0.6523
Epoch 4/8
45/45 ━━━━━━━━━━━━━━━━━━━━ 143s 3s/step - accuracy: 0.9235 - loss: 0.2616 - val_accuracy: 0.8315 - val_loss: 0.4656
Epoch 5/8
45/45 ━━━━━━━━━━━━━━━━━━━━ 140s 3s/step - accuracy: 0.9462 - loss: 0.1577 - val_accuracy: 0.8652 - val_loss: 0.5096
Epoch 6/8
45/45 ━━━━━━━━━━━━━━━━━━━━ 140s 3s/step - accuracy: 0.9462 - loss: 0.1594 - val_accuracy: 0.8989 - val_loss: 0.3761
Epoch 7/8
45/45 ━━━━━━━━━━━━━━━━━━━━ 141s 3s/step - accuracy: 0.9632 - loss: 0.1122 - val_accuracy: 0.8315 - val_loss: 0.5409
Epoch 8/8
45/45 ━━━━━━━━━━━━━━━━━━━━ 136s 3s/step - accuracy: 0.9150 - loss: 0.2692 - val_accuracy: 0.8876 - val_loss:

In [20]:
student_name = "Nora_Abdullah Aljomuh"
save_path = f"{student_name}_ucf11_model.h5"

model.save(save_path)
print(f"Model saved as {save_path}")

Model saved as Nora_Abdullah Aljomuh_ucf11_model.h5


In [22]:
def predict_video(video_path):
    frames = extract_frames(video_path)
    frames = np.expand_dims(frames, axis=0)

    prediction = model.predict(frames)
    predicted_class = np.argmax(prediction)

    print("Predicted Action:", CLASSES[predicted_class])
    print("Confidence:", np.max(prediction))

In [29]:
predict_video("/content/basketball.mp4")
predict_video("/content/bike.mp4")
predict_video("/content/diving.mp4")

Frames shape: (20, 64, 64, 3)
Frames sum: 34591.08235294119
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
Video: /content/basketball.mp4
Predicted Action: biking
Confidence: 0.7347636
All probabilities: [[0.26170942 0.7347636  0.00352701]]
----------------------------------------
Frames shape: (20, 64, 64, 3)
Frames sum: 120797.21176470586
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Video: /content/bike.mp4
Predicted Action: biking
Confidence: 0.81892943
All probabilities: [[0.16494027 0.81892943 0.01613034]]
----------------------------------------
Frames shape: (20, 64, 64, 3)
Frames sum: 120793.53333333334
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
Video: /content/diving.mp4
Predicted Action: diving
Confidence: 0.9932359
All probabilities: [[0.00122771 0.00553636 0.9932359 ]]
----------------------------------------
